In [59]:
import os
import json
import re
from pathlib import Path
from collections import OrderedDict
import xml.etree.ElementTree as ET

# ── Local Workspace Target Configuration ───────────────────────
WORKSPACE_DIR = Path("/Users/gcrane/Downloads/gemsite/classical_workspace3")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

NS = {'tei': 'http://www.tei-c.org/ns/1.0'}

WORK_REGISTRY = {
    "tlg0003.tlg001": {
        "textgroup": "tlg0003",
        "work": "tlg001",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-grc2.xml", "label": "Greek (H. S. Jones, 1942)", "class": "greek-text"},
            "1st1K-eng1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1K-eng1.xml", "label": "English (C. F. Smith, 1919)", "class": "english-text"},
            "perseus-eng6": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.perseus-eng6.xml", "label": "English (R. Crawley, 1914)", "class": "english-text"},
            "1st1k-fre1": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-fre1.xml", "label": "French (E. Bétant, 1863)", "class": "french-text"},
            "1st1k-lat2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0003/tlg001/tlg0003.tlg001.1st1k-lat2.xml", "label": "Latin (Fr. Haase, 1869)", "class": "latin-text"}
        }
    },
    "tlg0086.tlg034": {
        "textgroup": "tlg0086",
        "work": "tlg034",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-grc2.xml", "label": "Greek (Kassel, 1965)", "class": "greek-text"},
            "digicorpus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.digicorpus-grc2.xml", "label": "Greek (Digital Corpus Variant)", "class": "greek-text"},
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0086/tlg034/tlg0086.tlg034.perseus-eng2.xml", "label": "English (W.H. Fyfe, 1927)", "class": "english-text"},
            "butcher1911-eng2": {"path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.butcher1911-eng2.xml", "label": "English (S.H. Butcher, 1911)", "class": "english-text"},
            "bywater1909-eng1": {"path": "/Users/gcrane/github/Poetics2.0/grc/tlg0086.tlg034.bywater1909-eng1.xml", "label": "English (Ingram Bywater, 1909)", "class": "english-text"},
            "twining1789-eng1": {"path": "/Users/gcrane/github/Poetics2.0/eng/tlg0086.tlg034.twining1789-eng1.xml", "label": "English (Thomas Twining, 1789)", "class": "english-text", "parse_mode": "milestones"}
        }
    },
    "tlg0011.tlg004": {
        "textgroup": "tlg0011",
        "work": "tlg004",
        "editions": {
            "perseus-grc2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg004/tlg0011.tlg004.perseus-grc2.xml", "label": "Greek (F. Storr, 1912)", "class": "greek-text", "parse_mode": "poetry_cards"},
            "perseus-eng2": {"path": "/Users/gcrane/github/canonical-greekLit/data/tlg0011/tlg004/tlg0011.tlg004.perseus-eng2.xml", "label": "English (Sir Richard Jebb, 1904)", "class": "english-text", "parse_mode": "poetry_cards"}
        }
    }
}

def find_text_root(root):
    for div in root.findall('.//{http://www.tei-c.org/ns/1.0}div') + root.findall('.//div'):
        if div.get('type') == 'translation': return div
    return root.find('.//{http://www.tei-c.org/ns/1.0}body') or root.find('.//body') or root.find('.//*body')

def extract_text_recursive(elem):
    parts = []
    tag = elem.tag.split('}')[-1]
    
    if tag == 'l':
        line_num = (elem.get('n') or '').strip()
        parts.append(f'<div class="verse-line" data-line="{line_num}">')
        # We no longer add dynamic gutter numbers here because they are already present as text nodes in the source XML
    elif tag == 'speaker': 
        parts.append('<strong class="speaker-attr">')
    elif tag == 'stage':
        parts.append('<div class="stage-direction">')
    elif tag == 'hi':
        rend = elem.get('rend', 'italic')
        parts.append(f'<span class="render-{rend}">')
    elif tag == 'quote':
        q_type = elem.get('type', 'blockquote')
        parts.append(f'<blockquote class="quote-block type-{q_type}">')

    if elem.text: parts.append(elem.text)
    for child in elem:
        child_tag = child.tag.split('}')[-1]
        if child_tag == 'note':
            note_text = extract_text_recursive(child).strip()
            if note_text: parts.append(f'<span class="note">[{note_text}]</span>')
        elif child_tag == 'lb': parts.append('<br/>')
        else: parts.append(extract_text_recursive(child))
        if child.tail: parts.append(child.tail)
            
    if tag == 'l': parts.append('</div>')
    elif tag == 'speaker': parts.append(': </strong>')
    elif tag == 'stage': parts.append('</div>')
    elif tag == 'hi': parts.append('</span>')
    elif tag == 'quote': parts.append('</blockquote>')
    return ''.join(parts)

def parse_hierarchical_tei(path):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    root = tree.getroot()
    text_entry = find_text_root(root)
    if text_entry is None: return None
    data = OrderedDict()

    def walk_divisions(node, current_path):
        tag = node.tag.split('}')[-1]
        subtype = node.get('subtype') or node.get('type')
        n_val = node.get('n')

        if tag == 'div' and subtype in ('book', 'chapter', 'section', 'part', 'textpart') and n_val:
            resolved_subtype = 'chapter' if subtype in ('part', 'textpart') and n_val.isdigit() else subtype
            new_path = current_path + [(resolved_subtype, str(n_val).strip())]
        else:
            new_path = current_path

        paragraphs = node.findall('{http://www.tei-c.org/ns/1.0}p') or node.findall('p')
        if paragraphs and new_path:
            key_map = {t: v for t, v in new_path}
            bk = key_map.get('book', '1')
            ch = key_map.get('chapter', key_map.get('part', '1'))
            sec = key_map.get('section', n_val or '1')

            if bk not in data: data[bk] = OrderedDict()
            if ch not in data[bk]: data[bk][ch] = OrderedDict()
            
            combined_txt = ' '.join(extract_text_recursive(p).strip() for p in paragraphs)
            if combined_txt: data[bk][ch][sec] = combined_txt
            return

        for child in node: walk_divisions(child, new_path)

    walk_divisions(text_entry, [])
    return data

def parse_milestone_aligned_tei(path):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    root = tree.getroot()
    text_entry = find_text_root(root)
    if text_entry is None: return None

    data = OrderedDict()
    current_ref = "1.1"  
    current_text = []

    def _flush():
        nonlocal current_ref, current_text
        if current_text:
            if '.' in current_ref: ch, sec = current_ref.split('.', 1)
            else: ch, sec = current_ref, '1'
            bk = '1'
            data.setdefault(bk, OrderedDict()).setdefault(ch, OrderedDict())
            text = ' '.join(t.strip() for t in current_text if t.strip())
            if text:
                if sec in data[bk][ch]: data[bk][ch][sec] += " " + text
                else: data[bk][ch][sec] = text
        current_text = []

    def _walk(elem):
        nonlocal current_ref, current_text
        tag = elem.tag.split('}')[-1]

        if tag == 'milestone':
            val = elem.get('n') or elem.get('unit') or ''
            if re.match(r'\\d+', val):
                _flush()
                current_ref = str(val).strip()
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if tag == 'note':
            note_html = extract_text_recursive(elem).strip()
            if note_html: current_text.append(f'<span class="note">[{note_html}]</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if tag in ('hi', 'foreign', 'emph'):
            rend = elem.get('rend', 'italic')
            current_text.append(f'<span class="render-{rend}">')
            if elem.text: current_text.append(elem.text)
            for child in elem: _walk(child)
            current_text.append('</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if elem.text and elem.text.strip(): current_text.append(elem.text)
        for child in elem: _walk(child)
        if elem.tail and elem.tail.strip(): current_text.append(elem.tail)

    _walk(text_entry)
    _flush()  
    return data

def build_poetry_canonical_intervals(editions_dict):
    grc_cfg = editions_dict.get('perseus-grc2') or list(editions_dict.values())[0]
    tree = ET.parse(grc_cfg["path"])
    text_entry = find_text_root(tree.getroot())
    
    landmarks = []
    for elem in text_entry.iter():
        if elem.tag.endswith('milestone') and elem.get('unit') == 'card':
            landmarks.append(('card', elem.get('n').strip()))
        elif elem.tag.endswith('l'):
            ln = (elem.get('n') or '').strip()
            if ln: landmarks.append(('line', ln))

    intervals = []
    for idx, item in enumerate(landmarks):
        if item[0] == 'card':
            card_n = item[1]
            first_l, last_l = None, None
            for ahead in landmarks[idx+1:]:
                if ahead[0] == 'card': break
                if ahead[0] == 'line':
                    if first_l is None: first_l = ahead[1]
                    last_l = ahead[1]
            if not first_l: first_l = card_n
            if not last_l: last_l = first_l
            intervals.append({'card_n': card_n, 'label': f"{first_l}-{last_l}"})
    return intervals

def parse_poetry_cards_tei(path, master_intervals):
    if not os.path.exists(path): return None
    tree = ET.parse(path)
    text_entry = find_text_root(tree.getroot())

    data = OrderedDict()
    data["1"] = OrderedDict()
    
    current_label = master_intervals[0]['label'] if master_intervals else "1"
    current_text = []

    def _flush():
        nonlocal current_label, current_text
        if current_text and current_label:
            html = ' '.join(t.strip() for t in current_text if t.strip())
            if html:
                if current_label not in data["1"]:
                    data["1"][current_label] = OrderedDict()
                data["1"][current_label]["1"] = html
        current_text = []

    def _walk(elem):
        nonlocal current_label, current_text
        tag = elem.tag.split('}')[-1]

        if tag == 'milestone' and elem.get('unit') == 'card':
            val = (elem.get('n') or '').strip()
            match = [r for r in master_intervals if r['card_n'] == val]
            if match:
                _flush()
                current_label = match[0]['label']
        elif tag == 'l' or tag == 'stage':
            current_text.append(extract_text_recursive(elem))
            return
        elif tag == 'note':
            note_html = extract_text_recursive(elem).strip()
            if note_html: current_text.append(f'<span class="note">[{note_html}]</span>')
            if elem.tail and elem.tail.strip(): current_text.append(elem.tail)
            return

        if elem.text and elem.text.strip(): current_text.append(elem.text)
        for child in elem: _walk(child)
        if elem.tail and elem.tail.strip(): current_text.append(elem.tail)

    _walk(text_entry)
    _flush()
    return data

GLOBAL_STRUCTURES = {}
GLOBAL_REGISTRIES = {}

for work_key, work_meta in WORK_REGISTRY.items():
    tg = work_meta["textgroup"]
    wk = work_meta["work"]
    editions = work_meta["editions"]
    
    print(f"Ingesting textual data layers for canonical identifier context: {work_key}...")
    
    for v_id, cfg in editions.items():
        GLOBAL_REGISTRIES[v_id] = {
            "urn": f"urn:cts:greekLit:{work_key}.{v_id}",
            "label": cfg["label"],
            "class": cfg["class"],
            "textgroup": tg,
            "work": wk
        }

    master_intervals = None
    is_poetry = any(cfg.get("parse_mode") == "poetry_cards" for cfg in editions.values())
    if is_poetry:
        master_intervals = build_poetry_canonical_intervals(editions)

    work_corpus = OrderedDict()
    for v_id, cfg in editions.items():
        p_mode = cfg.get("parse_mode")
        if p_mode == "poetry_cards":
            parsed = parse_poetry_cards_tei(cfg["path"], master_intervals)
        elif p_mode == "milestones":
            parsed = parse_milestone_aligned_tei(cfg["path"])
        else:
            parsed = parse_hierarchical_tei(cfg["path"])
            
        if parsed is not None and sum(len(secs) for chs in parsed.values() for secs in chs.values()) > 0:
            work_corpus[v_id] = parsed
            print(f"  ✓ {v_id}: {sum(len(secs) for chs in parsed.values() for secs in chs.values())} segments parsed")
        else:
            print(f"  ✗ {v_id}: Failed to parse completely.")

    if not work_corpus: continue

    first_version = list(work_corpus.keys())[0]
    baseline_corpus = work_corpus[first_version]

    has_multiple_books = len(baseline_corpus.keys()) > 1
    structure_map = OrderedDict()
    chapter_sequence = []

    if has_multiple_books:
        for b_k, ch_v in baseline_corpus.items():
            structure_map[b_k] = list(ch_v.keys())
            for c_k in ch_v.keys():
                chapter_sequence.append({'book': b_k, 'chapter': c_k})
        GLOBAL_STRUCTURES[work_key] = structure_map
    else:
        single_bk_key = list(baseline_corpus.keys())[0]
        structure_map["_flat_chapters"] = list(baseline_corpus[single_bk_key].keys())
        for c_k in baseline_corpus[single_bk_key].keys():
            chapter_sequence.append({'book': None, 'chapter': c_k})
        GLOBAL_STRUCTURES[work_key] = structure_map["_flat_chapters"]

    print(f" -> Packaging structural formatted JS chunk scripts...")
    for c_idx, coord in enumerate(chapter_sequence):
        bk_id = coord['book']
        ch_id = coord['chapter']
        
        lookup_bk = bk_id if bk_id else list(baseline_corpus.keys())[0]
        baseline_secs = list(baseline_corpus[lookup_bk][ch_id].keys())
        sections_payload = OrderedDict()
        
        for sec in baseline_secs:
            sections_payload[sec] = {}
            for v_id in editions:
                ch_data = work_corpus.get(v_id, {}).get(lookup_bk, {}).get(ch_id, {})
                sections_payload[sec][v_id] = ch_data.get(sec, "<i>[Text range missing in alignment layer]</i>")

        if bk_id:
            passage_urn = f"urn:cts:greekLit:{work_key}:{bk_id}.{ch_id}"
            prev_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx-1]['book']}.{chapter_sequence[c_idx-1]['chapter']}" if c_idx > 0 else None
            next_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx+1]['book']}.{chapter_sequence[c_idx+1]['chapter']}" if c_idx < len(chapter_sequence) - 1 else None
            chunk_filename = f"chunk_b{bk_id}_ch{ch_id}.js"
        else:
            passage_urn = f"urn:cts:greekLit:{work_key}:{ch_id}"
            prev_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx-1]['chapter']}" if c_idx > 0 else None
            next_urn = f"urn:cts:greekLit:{work_key}:{chapter_sequence[c_idx+1]['chapter']}" if c_idx < len(chapter_sequence) - 1 else None
            chunk_filename = f"chunk_ch{ch_id}.js"
        
        chapter_payload = {
            "urn": passage_urn,
            "textgroup": tg,
            "work": wk,
            "book": bk_id,
            "chapter": ch_id,
            "sections": sections_payload,
            "navigation": { "prev": prev_urn, "next": next_urn }
        }

        chunk_dir = WORKSPACE_DIR / "corpus" / tg / wk / "chunks"
        chunk_dir.mkdir(parents=True, exist_ok=True)
        
        js_wrapped = f"""/** Perseus Autonomous Text Chunk Module **/
registerWorkspaceChunk("{passage_urn}", {json.dumps(chapter_payload, indent=2, ensure_ascii=False)});
"""
        (chunk_dir / chunk_filename).write_text(js_wrapped, encoding='utf-8')

print("[SUCCESS] Data Ingestion Engine completed cleanly for all works.")

Ingesting textual data layers for canonical identifier context: tlg0003.tlg001...
  ✓ perseus-grc2: 3587 segments parsed
  ✓ 1st1K-eng1: 3580 segments parsed
  ✓ perseus-eng6: 3587 segments parsed
  ✓ 1st1k-fre1: 917 segments parsed
  ✓ 1st1k-lat2: 3624 segments parsed
 -> Packaging structural formatted JS chunk scripts...
Ingesting textual data layers for canonical identifier context: tlg0086.tlg034...
  ✓ perseus-grc2: 381 segments parsed
  ✓ digicorpus-grc2: 381 segments parsed
  ✓ perseus-eng2: 379 segments parsed
  ✓ butcher1911-eng2: 381 segments parsed
  ✓ bywater1909-eng1: 380 segments parsed
  ✓ twining1789-eng1: 1 segments parsed
 -> Packaging structural formatted JS chunk scripts...
Ingesting textual data layers for canonical identifier context: tlg0011.tlg004...
  ✓ perseus-grc2: 72 segments parsed
  ✓ perseus-eng2: 70 segments parsed
 -> Packaging structural formatted JS chunk scripts...
[SUCCESS] Data Ingestion Engine completed cleanly for all works.


In [61]:
struct_map_json = json.dumps(GLOBAL_STRUCTURES)
text_registry_json = json.dumps(GLOBAL_REGISTRIES)

INDEX_HTML_CONTENT = """<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Global Core Parallel Reading Workspace</title>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        html, body {
            height: 100%; width: 100%; overflow: hidden; 
            font-family: "Palatino Linotype", "Book Antiqua", Palatino, Georgia, serif;
            font-size: 13px; color: #000; background: #fff;
        }
        a { color: #336699; text-decoration: none; }
        a:hover { text-decoration: underline; }

        #app-view-root { display: flex; flex-direction: column; height: 100vh; width: 100vw; overflow: hidden; }

        #header-container { flex-shrink: 0; background: #fff; border-bottom: 1px solid #ccc; z-index: 10; }
        #perseus-banner { background: #660000; color: #fff; padding: 6px 15px; display: flex; justify-content: space-between; align-items: center; }
        #perseus-banner h1 a { color: #fff; font-size: 16px; font-weight: bold; }
        #perseus-banner .doc-title { font-size: 11px; color: #ffccaa; font-weight: bold; }

        #nav-bar { background: #ddddcc; border-bottom: 1px solid #999988; padding: 4px 15px; font-size: 11px; font-weight: bold; }

        #browse-bar { background: #eeeeee; border-bottom: 1px solid #cccccc; padding: 6px 15px; font-size: 11px; display: flex; flex-direction: column; gap: 5px; }
        .browse-row { display: flex; align-items: flex-start; }
        .browse-row.hidden-row { display: none !important; }
        .browse-row label { font-weight: bold; width: 80px; color: #444; flex-shrink: 0; padding-top: 1px; }
        .browse-items { display: flex; flex-wrap: wrap; gap: 4px; }
        .browse-items a { padding: 1px 6px; background: #e0e0d0; color: #336699; border: 1px solid #bbbb99; font-weight: bold; border-radius: 2px; cursor: pointer; }
        .browse-items a.current { background: #660000; color: #fff; border-color: #330000; }

        .browse-items a.sec-pill { background: #fff; color: #555; border: 1px solid #ccc; font-weight: normal; font-size: 10px; padding: 1px 5px; }
        .browse-items a.sec-pill.active-pill { background: #660000; color: #fff; border-color: #330000; font-weight: bold; }

        #outer-wrapper { display: flex; width: 100%; flex: 1; min-height: 0; }

        #main-container { flex: 1; display: flex; min-width: 0; height: 100%; }
        .reading-column { display: flex; flex-direction: column; flex: 1; min-width: 0; height: 100%; border-right: 1px solid #eeeeee; background: #fff; }
        .reading-column:last-child { border-right: none; }
        .cross-panel { background: #fafafa; }

        .panel-header { background: #660000; color: #fff; padding: 4px 8px; font-size: 11px; font-weight: bold; position: sticky; top: 0; z-index: 5; display: flex; justify-content: space-between; align-items: center; flex-shrink: 0; }
        .edition-select-dropdown { background: #fff; color: #333; font-size: 10px; padding: 1px 4px; border: 1px solid #ccc; border-radius: 2px; outline: none; cursor: pointer; }

        .column-content-scroll { flex: 1; overflow-y: auto; padding: 15px; }

        /* ── Core Layout Rows ── */
        .section-row { 
            border-bottom: 1px solid #f0f0f0; 
            padding: 10px 0; 
            display: block;
            scroll-margin-top: 5px; 
        }
        .section-row.hidden-section { display: none !important; }
        
        .section-row .sec-num { 
            font-weight: bold; 
            color: #990000; 
            width: 55px; 
            display: inline-block;
            vertical-align: top;
            padding-top: 1px;
        }

        .greek-text { font-size: 15px; line-height: 1.7; font-family: "Gentium Plus", "Athena", serif; color: #000; }
        .english-text { font-size: 13px; line-height: 1.6; color: #222; }
        .french-text { font-size: 13px; line-height: 1.6; color: #2b2b2b; }
        .latin-text { font-size: 13px; line-height: 1.6; color: #111; font-style: italic; }

        /* ── Poetry CSS Grid Structure ── */
        .poetry-grid-layout {
            display: grid !important;
            grid-template-columns: 45px 1fr;
            row-gap: 4px;
            align-items: start;
            width: 100%;
        }

        /* Forces left column tracking to stay perfectly square on a single row */
        .poetry-grid-layout .line-num-cell {
            grid-column: 1;
            text-align: right; 
            padding-right: 12px;
            font-size: 11px; 
            color: #999; 
            font-weight: bold; 
            font-family: "Courier New", Courier, monospace;
            line-height: 1.7;
            user-select: none;
            display: block !important;
        }

        /* Forces line body text to sit directly adjacent to the number */
        .poetry-grid-layout .line-text-cell {
            grid-column: 2;
            line-height: 1.7;
            display: block !important;
            white-space: normal;
        }

        /* ── Poetry Stage Directions ── */
        /* Breaks stage descriptions out of columns to span the full text width in italics */
        .poetry-grid-layout .stage-direction {
            grid-column: 1 / span 2 !important;
            display: block !important;
            font-style: italic;
            color: #445566;
            background: #fdfdfd;
            border-left: 3px solid #ccc;
            padding: 8px 12px;
            margin: 10px 0;
            line-height: 1.6;
            font-size: 0.95em;
        }

        /* Prose Layout */
        .prose-inline-layout {
            display: block;
            width: 100%;
            line-height: 1.6;
        }
        .prose-inline-layout .prose-marker {
            font-weight: bold; 
            color: #990000; 
            margin-right: 8px;
            display: inline;
        }
        .prose-inline-layout .prose-body {
            display: inline;
        }

        .speaker-attr { color: #445566; font-variant: small-caps; letter-spacing: 0.5px; display: block; margin-top: 8px; margin-bottom: 4px; }
        .poetry-grid-layout .speaker-attr { grid-column: 1 / span 2; display: block !important; }
        
        .render-italic, .render-italics { font-style: italic; }
        .render-bold { font-weight: bold; }
        .render-underline { text-decoration: underline; }
        
        .note { font-size: 11px; color: #555; background: #fdfbf7; border-left: 2px solid #880000; padding: 2px 6px; margin: 4px 0 4px 1em; display: block; }
        .poetry-grid-layout .note { grid-column: 1 / span 2; display: block !important; }

        .quote-block {
            margin: 10px 0 10px 2.5em;
            padding-left: 12px;
            border-left: 1px dashed #bbb;
            font-size: 0.95em;
            color: #333;
            grid-column: 1 / span 2;
        }
        .quote-block.type-verse { font-style: italic; line-height: 1.6; background: #fafafa; padding: 6px 12px; }
        .quote-block.type-blockquote { line-height: 1.55; color: #1a1a1a; }

        .viewport-footer-controls { display: flex; justify-content: space-between; align-items: center; gap: 8px; margin-top: 20px; padding-top: 10px; border-top: 1px solid #ddd; width: 100%; clear: both; }
        .poetry-grid-layout .viewport-footer-controls { grid-column: 1 / span 2; display: flex !important; }
        .footer-group-left { display: flex; gap: 6px; }
        .action-btn { display: inline-block; background: #660000; color: #fff !important; padding: 4px 10px; font-size: 11px; font-weight: bold; border-radius: 2px; border: none; cursor: pointer; }
        .action-btn.secondary { background: #666; }
        .action-btn:hover { background: #330000; text-decoration: none; }
        .action-btn.secondary:hover { background: #444; }
    </style>
</head>
<body>

  <div id="app-view-root">
    <div id="header-container">
      <div id="perseus-banner">
        <h1><a href="#">Perseus Workspace Engine</a></h1>
        <span class="doc-title">Global CTS-URN Unified Environment</span>
      </div>
      <div id="nav-bar">
        <span id="frame-context-label">Loading system configuration modules...</span>
      </div>
      
      <div id="browse-bar">
        <div class="browse-row">
          <label>Works:</label>
          <div class="browse-items" id="work-items-container"></div>
        </div>
        <div class="browse-row" id="row-book-container">
          <label>Books:</label>
          <div class="browse-items" id="book-items-container"></div>
        </div>
        <div class="browse-row">
          <label id="chapter-row-label">Chapters:</label>
          <div class="browse-items" id="chapter-items-container"></div>
        </div>
        <div class="browse-row">
          <label>Sections:</label>
          <div class="browse-items" id="section-items-container"></div>
        </div>
      </div>
    </div>

    <div id="outer-wrapper">
      <div id="main-container">
        <div class="reading-column" id="col_f">
            <div class="panel-header">
                <span>Focal Axis Text Column</span>
                <select class="edition-select-dropdown" id="select_f" onchange="updateColumnContent('f', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_f"></div>
        </div>
        <div class="reading-column cross-panel" id="col_c1">
            <div class="panel-header">
                <span>Comparison Version 1</span>
                <select class="edition-select-dropdown" id="select_c1" onchange="updateColumnContent('c1', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_c1"></div>
        </div>
        <div class="reading-column cross-panel" id="col_c2">
            <div class="panel-header">
                <span>Comparison Version 2</span>
                <select class="edition-select-dropdown" id="select_c2" onchange="updateColumnContent('c2', this.value)"></select>
            </div>
            <div class="column-content-scroll" id="content_c2"></div>
        </div>
      </div>
    </div>
  </div>

  <script>
    const GLOBAL_STRUCTURES = STRUCT_REPLACE;
    const TEXT_REGISTRY = REGISTRY_REPLACE;
    
    const WORKSPACE_DATA_REGISTRY = new Map();
    
    let activeWorkKey = "";
    let activeUrnContext = "";
    let activeSectionFilter = null;

    let columnEditions = { f: "", c1: "", c2: "" };

    window.registerWorkspaceChunk = function(urn, dataPayload) {
        WORKSPACE_DATA_REGISTRY.set(urn, dataPayload);
        triggerRenderLifecycle(urn);
    };

    document.addEventListener("DOMContentLoaded", function() {
        initializeRoutingFromURL();
    });

    function populateDropdownsForWork(tgId, wkId) {
        const validEditionIds = Object.keys(TEXT_REGISTRY).filter(vId => {
            return TEXT_REGISTRY[vId].textgroup === tgId && TEXT_REGISTRY[vId].work === wkId;
        });

        if (validEditionIds.length === 0) return;

        if (!columnEditions.f || !validEditionIds.includes(columnEditions.f)) columnEditions.f = validEditionIds[0];
        if (!columnEditions.c1 || !validEditionIds.includes(columnEditions.c1)) columnEditions.c1 = validEditionIds[1] || validEditionIds[0];
        if (!columnEditions.c2 || !validEditionIds.includes(columnEditions.c2)) columnEditions.c2 = validEditionIds[2] || validEditionIds[0];

        ['f', 'c1', 'c2'].forEach(prefix => {
            const selectEl = document.getElementById(`select_${prefix}`);
            if (!selectEl) return;
            selectEl.innerHTML = "";
            
            validEditionIds.forEach(vId => {
                const opt = document.createElement("option");
                opt.value = vId;
                opt.innerText = TEXT_REGISTRY[vId].label;
                if(columnEditions[prefix] === vId) opt.selected = true;
                selectEl.appendChild(opt);
            });
        });
    }

    function isFlatStructure(wKey) {
        return Array.isArray(GLOBAL_STRUCTURES[wKey]);
    }

    function isPoetryWork(wKey) {
        return wKey.startsWith("tlg0011.");
    }

    function initializeRoutingFromURL() {
        try {
            const params = new URLSearchParams(window.location.search);
            const rawParam = params.get("w") || "";
            
            let b = "";
            let ch = "";
            activeSectionFilter = null;
            
            columnEditions.c1 = params.get("right") || "";
            columnEditions.c2 = params.get("right2") || "";

            if (rawParam.includes(":")) {
                const querySegments = rawParam.split(":");
                let workPart = querySegments[0];
                
                const workDots = workPart.split(".");
                if (workDots.length > 2) {
                    const potentialEdition = workDots.pop();
                    const fallbackBaseKey = workDots.join(".");
                    if (TEXT_REGISTRY[potentialEdition]) {
                        columnEditions.f = potentialEdition;
                        workPart = fallbackBaseKey;
                    }
                } else {
                    columnEditions.f = "";
                }
                
                activeWorkKey = workPart;
                const passageSegments = querySegments[1].split(".");
                
                if (isFlatStructure(activeWorkKey)) {
                    b = ""; 
                    ch = passageSegments[0] || "1";
                    if (passageSegments[1]) activeSectionFilter = passageSegments[1];
                } else {
                    b = passageSegments[0];
                    ch = passageSegments[1] || "1";
                    if (passageSegments[2]) activeSectionFilter = passageSegments[2];
                }
            } else {
                activeWorkKey = rawParam || Object.keys(GLOBAL_STRUCTURES)[0];
                if (!GLOBAL_STRUCTURES[activeWorkKey]) {
                    activeWorkKey = Object.keys(GLOBAL_STRUCTURES)[0];
                }
                
                if (isFlatStructure(activeWorkKey)) {
                    b = "";
                    ch = GLOBAL_STRUCTURES[activeWorkKey][0] || "1";
                } else {
                    b = Object.keys(GLOBAL_STRUCTURES[activeWorkKey])[0];
                    ch = GLOBAL_STRUCTURES[activeWorkKey][b][0] || "1";
                }
                columnEditions.f = "";
            }

            const targetUrn = b ? `urn:cts:greekLit:${activeWorkKey}:${b}.${ch}` : `urn:cts:greekLit:${activeWorkKey}:${ch}`;
            resolveAndInjectUrn(targetUrn);
        } catch (err) {
            console.error("Initialization Failed:", err);
            document.getElementById("frame-context-label").innerText = "Runtime Error: " + err.message;
        }
    }

    function resolveAndInjectUrn(urn) {
        if (WORKSPACE_DATA_REGISTRY.has(urn)) {
            triggerRenderLifecycle(urn);
            return;
        }

        document.getElementById("frame-context-label").innerText = `Resolving storage path for ${urn}...`;
        
        const parts = urn.split(":");
        const workComponents = parts[3].split(".");
        const passageComponents = parts[4].split(".");
        
        const tgId = workComponents[0];
        const wkId = workComponents[1];

        populateDropdownsForWork(tgId, wkId);

        const script = document.createElement("script");
        if (passageComponents.length > 1 && !isFlatStructure(`${tgId}.${wkId}`)) {
            script.src = `corpus/${tgId}/${wkId}/chunks/chunk_b${passageComponents[0]}_ch${passageComponents[1]}.js`;
        } else {
            script.src = `corpus/${tgId}/${wkId}/chunks/chunk_ch${passageComponents[0]}.js`;
        }
        
        script.onerror = () => {
            document.getElementById("frame-context-label").innerText = `Path Resolution Failed for ${urn}`;
        };
        document.head.appendChild(script);
    }

    function triggerRenderLifecycle(urn) {
        activeUrnContext = urn;
        const payload = WORKSPACE_DATA_REGISTRY.get(urn);
        if (!payload) return;
        
        activeWorkKey = `${payload.textgroup}.${payload.work}`;
        populateDropdownsForWork(payload.textgroup, payload.work);

        updateURLState(payload.book, payload.chapter);
        renderNavigationControls(payload);
        renderActiveContentLayers(payload);
    }

    function updateURLState(book, chapter) {
        const params = new URLSearchParams();
        let passageValue = book ? `${book}.${chapter}` : `${chapter}`;
        if (activeSectionFilter) {
            passageValue += `.${activeSectionFilter}`;
        }
        
        params.set("w", `${activeWorkKey}.${columnEditions.f}:${passageValue}`);
        if(columnEditions.c1) params.set("right", columnEditions.c1);
        if(columnEditions.c2) params.set("right2", columnEditions.c2);
        
        const currentFilename = window.location.pathname.split('/').pop() || 'index.html';
        window.history.replaceState(null, "", currentFilename + "?" + params.toString());
    }

    function selectSectionDirectly(secId) {
        activeSectionFilter = secId;
        if (WORKSPACE_DATA_REGISTRY.has(activeUrnContext)) {
            triggerRenderLifecycle(activeUrnContext);
        }
    }

    function renderNavigationControls(payload) {
        document.getElementById("frame-context-label").innerText = `Active Frame Context: ${payload.urn}`;
        
        const workContainer = document.getElementById("work-items-container");
        if (workContainer) {
            workContainer.innerHTML = "";
            Object.keys(GLOBAL_STRUCTURES).forEach(wKey => {
                const a = document.createElement("a");
                if (wKey === "tlg0003.tlg001") a.innerText = "Thucydides (Histories)";
                else if (wKey === "tlg0086.tlg034") a.innerText = "Aristotle (Poetics)";
                else if (wKey === "tlg0011.tlg004") a.innerText = "Sophocles (Oedipus Tyrannus)";
                else a.innerText = wKey;
                
                if(wKey === activeWorkKey) a.className = "current";
                a.onclick = () => {
                    activeSectionFilter = null;
                    activeWorkKey = wKey;
                    if (isFlatStructure(wKey)) {
                        resolveAndInjectUrn(`urn:cts:greekLit:${wKey}:${GLOBAL_STRUCTURES[wKey][0]}`);
                    } else {
                        const targetBook = Object.keys(GLOBAL_STRUCTURES[wKey])[0];
                        resolveAndInjectUrn(`urn:cts:greekLit:${wKey}:${targetBook}.${GLOBAL_STRUCTURES[wKey][targetBook][0]}`);
                    }
                };
                workContainer.appendChild(a);
            });
        }

        const bookRow = document.getElementById("row-book-container");
        if(isFlatStructure(activeWorkKey)) {
            bookRow.classList.add("hidden-row");
        } else {
            bookRow.classList.remove("hidden-row");
            const bookContainer = document.getElementById("book-items-container");
            if (bookContainer) {
                bookContainer.innerHTML = "";
                Object.keys(GLOBAL_STRUCTURES[activeWorkKey]).forEach(bk => {
                    const a = document.createElement("a");
                    a.innerText = bk;
                    if(bk === payload.book) a.className = "current";
                    a.onclick = () => {
                        activeSectionFilter = null;
                        resolveAndInjectUrn(`urn:cts:greekLit:${activeWorkKey}:${bk}.${GLOBAL_STRUCTURES[activeWorkKey][bk][0]}`);
                    };
                    bookContainer.appendChild(a);
                });
            }
        }

        const bookLabelEl = document.getElementById("chapter-row-label");
        if (bookLabelEl) {
            bookLabelEl.innerText = isPoetryWork(activeWorkKey) ? "Ranges:" : "Chapters:";
        }

        const chapterContainer = document.getElementById("chapter-items-container");
        if (chapterContainer) {
            chapterContainer.innerHTML = "";
            const chList = isFlatStructure(activeWorkKey) ? GLOBAL_STRUCTURES[activeWorkKey] : GLOBAL_STRUCTURES[activeWorkKey][payload.book];
            chList.forEach(ch => {
                const a = document.createElement("a");
                a.innerText = ch;
                if(ch === payload.chapter) a.className = "current";
                a.onclick = () => {
                    activeSectionFilter = null;
                    const nextUrn = payload.book ? `urn:cts:greekLit:${activeWorkKey}:${payload.book}.${ch}` : `urn:cts:greekLit:${activeWorkKey}:${ch}`;
                    resolveAndInjectUrn(nextUrn);
                };
                chapterContainer.appendChild(a);
            });
        }

        const sectionContainer = document.getElementById("section-items-container");
        if (sectionContainer) {
            sectionContainer.innerHTML = "";
            Object.keys(payload.sections).forEach(sec => {
                const a = document.createElement("a");
                a.innerText = sec;
                a.className = "sec-pill";
                if(sec === activeSectionFilter) a.classList.add("active-pill");
                a.onclick = () => selectSectionDirectly(sec);
                sectionContainer.appendChild(a);
            });
        }
    }

    function renderActiveContentLayers(payload) {
        ['f', 'c1', 'c2'].forEach(prefix => {
            const targetContainer = document.getElementById(`content_${prefix}`);
            if (!targetContainer) return;
            targetContainer.innerHTML = "";
            
            const vId = columnEditions[prefix];
            const cssClass = TEXT_REGISTRY[vId] ? TEXT_REGISTRY[vId].class : "english-text";
            const isPoetry = isPoetryWork(activeWorkKey);
            
            Object.keys(payload.sections).forEach(sec => {
                const isHidden = activeSectionFilter !== null && activeSectionFilter !== sec;
                const row = document.createElement("div");
                row.className = `section-row s-idx-${sec} ${isHidden ? 'hidden-section' : ''}`;
                
                const txt = payload.sections[sec][vId] || "<i>[Text range missing in alignment layer]</i>";
                const visualIndexLabel = isPoetry ? sec : `[${sec}]`;
                const poetryClassSuffix = isPoetry ? " is-poetry-layout" : "";
                
                if (isPoetry) {
                    const wrapper = document.createElement("div");
                    wrapper.className = `${cssClass} poetry-grid-layout`;
                    
                    const temp = document.createElement("div");
                    temp.innerHTML = txt;
                    
                    while (temp.firstChild) {
                        const child = temp.firstChild;
                        if (child.nodeType === Node.ELEMENT_NODE && child.classList.contains("verse-line")) {
                            const lineNum = child.getAttribute("data-line") || "";
                            
                            const numCell = document.createElement("span");
                            numCell.className = "line-num-cell";
                            numCell.innerText = lineNum;
                            
                            const textCell = document.createElement("div");
                            textCell.className = "line-text-cell";
                            textCell.innerHTML = child.innerHTML;
                            
                            wrapper.appendChild(numCell);
                            wrapper.appendChild(textCell);
                            temp.removeChild(child);
                        } else {
                            if (child.nodeType === Node.ELEMENT_NODE) {
                                // Maps stage directions across the full row tracks dynamically
                                if (child.classList.contains("stage-direction")) {
                                    child.style.gridColumn = "1 / span 2";
                                    child.style.display = "block";
                                } else {
                                    child.style.gridColumn = "1 / span 2";
                                    child.style.display = "block";
                                }
                            }
                            wrapper.appendChild(child);
                        }
                    }
                    row.appendChild(wrapper);
                } else {
                    row.innerHTML = `
                        <div class="prose-inline-layout ${cssClass}">
                            <span class="prose-marker"><a href="javascript:void(0)" onclick="selectSectionDirectly('${sec}')">${visualIndexLabel}</a></span>
                            <div class="prose-body">${txt}</div>
                        </div>
                    `;
                }
                
                targetContainer.appendChild(row);
            });

            const footer = document.createElement("div");
            footer.className = "viewport-footer-controls";
            
            let prevBtnHtml = "";
            let nextBtnHtml = "";
            
            if (activeSectionFilter) {
                const secArray = Object.keys(payload.sections);
                const currentIdx = secArray.indexOf(activeSectionFilter);
                
                if (currentIdx > 0) {
                    prevBtnHtml = `<a onclick="selectSectionDirectly('${secArray[currentIdx-1]}')" class="action-btn">&larr; Previous [${secArray[currentIdx-1]}]</a>`;
                } else if (payload.navigation.prev) {
                    prevBtnHtml = `<a class="action-btn" onclick="loadAdjacentUrn('${payload.navigation.prev}', true)">&larr; Previous Segment</a>`;
                }
                
                if (currentIdx < secArray.length - 1) {
                    nextBtnHtml = `<a onclick="selectSectionDirectly('${secArray[currentIdx+1]}')" class="action-btn">Next [${secArray[currentIdx+1]}] &rarr;</a>`;
                } else if (payload.navigation.next) {
                    nextBtnHtml = `<a class="action-btn" onclick="loadAdjacentUrn('${payload.navigation.next}', false)">Next Segment &rarr;</a>`;
                }
            } else {
                if (payload.navigation.prev) {
                    prevBtnHtml = `<a class="action-btn" onclick="loadAdjacentUrn('${payload.navigation.prev}')">&larr; Previous Segment</a>`;
                }
                if (payload.navigation.next) {
                    nextBtnHtml = `<a class="action-btn" onclick="loadAdjacentUrn('${payload.navigation.next}')">Next Chapter &rarr;</a>`;
                }
            }

            footer.innerHTML = `
                <div class="footer-group-left">
                    ${prevBtnHtml}
                    <a href="javascript:void(0)" class="action-btn secondary" onclick="clearSectionFilter(event)">Full View</a>
                </div>
                ${nextBtnHtml}
            `;
            
            const targetGrid = targetContainer.querySelector(".poetry-grid-layout");
            if (targetGrid && isPoetry) {
                targetGrid.appendChild(footer);
            } else {
                targetContainer.appendChild(footer);
            }
        });
    }

    window.loadAdjacentUrn = function(urn, selectLastSection = false) {
        activeSectionFilter = null;
        if (selectLastSection) {
            if (WORKSPACE_DATA_REGISTRY.has(urn)) {
                const data = WORKSPACE_DATA_REGISTRY.get(urn);
                const secs = Object.keys(data.sections);
                activeSectionFilter = secs.length > 0 ? secs[secs.length - 1] : null;
                triggerRenderLifecycle(urn);
            } else {
                const parts = urn.split(":");
                const workComponents = parts[3].split(".");
                const passageComponents = parts[4].split(".");
                const script = document.createElement("script");
                
                if (passageComponents.length > 1 && !isFlatStructure(`${workComponents[0]}.${workComponents[1]}`)) {
                    script.src = `corpus/${workComponents[0]}/${workComponents[1]}/chunks/chunk_b${passageComponents[0]}_ch${passageComponents[1]}.js`;
                } else {
                    script.src = `corpus/${workComponents[0]}/${workComponents[1]}/chunks/chunk_ch${passageComponents[0]}.js`;
                }
                
                script.onload = () => {
                    const data = WORKSPACE_DATA_REGISTRY.get(urn);
                    const secs = Object.keys(data.sections);
                    activeSectionFilter = secs.length > 0 ? secs[secs.length - 1] : null;
                    triggerRenderLifecycle(urn);
                };
                document.head.appendChild(script);
            }
        } else {
            resolveAndInjectUrn(urn);
        }
    };

    window.clearSectionFilter = function(e) {
        if(e) e.preventDefault();
        activeSectionFilter = null;
        if (WORKSPACE_DATA_REGISTRY.has(activeUrnContext)) {
            triggerRenderLifecycle(activeUrnContext);
        }
    };

    window.updateColumnContent = function(prefix, value) {
        columnEditions[prefix] = value;
        if (activeUrnContext && WORKSPACE_DATA_REGISTRY.has(activeUrnContext)) {
            const payload = WORKSPACE_DATA_REGISTRY.get(activeUrnContext);
            updateURLState(payload.book, payload.chapter);
            renderActiveContentLayers(payload);
        }
    };
  </script>
</body>
</html>
""".replace("STRUCT_REPLACE", struct_map_json).replace("REGISTRY_REPLACE", text_registry_json)

(WORKSPACE_DIR / "index.html").write_text(INDEX_HTML_CONTENT, encoding='utf-8')
print("[SUCCESS] Core dashboard frontend compiled inside classical_workspace3.")

[SUCCESS] Core dashboard frontend compiled inside classical_workspace3.
